In [1]:
import os
import re
import string
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Phase 1: Environment Setup

In [2]:
for resource in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    try:
        nltk.data.find(resource)
    except LookupError:
        nltk.download(resource, quiet=True)

STOPWORDS = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

DATASET_PATH = "../dataset/sentiment_data.csv"
BACKEND_DIR = "../backend" 

# Phase 2: Dataset Exploration

In [11]:
DATASET_PATH="../dataset/sentiment_data.csv"  
BACKEND_DIR="../backend"

df=pd.read_csv(DATASET_PATH)

if "Unnamed: 0" in df.columns:
    df=df.drop(columns=["Unnamed: 0"])

df=df.rename(columns={
    "Comment":"text",
    "Sentiment":"sentiment"
})

label_map={
    0:"Negative",
    1:"Neutral",
    2:"Positive"
}
df["sentiment"]=df["sentiment"].map(label_map)

display(df.head())
print(df.columns.tolist())
print(df.shape)
print(df["sentiment"].value_counts())
print(df.isnull().sum())

,text,sentiment
0,lets forget apple pay required brand new iphon...,Neutral
1,nz retailers don’t even contactless credit car...,Negative
2,forever acknowledge channel help lessons ideas...,Positive
3,whenever go place doesn’t take apple pay doesn...,Negative
4,apple pay convenient secure easy use used kore...,Positive


['text', 'sentiment']
(241145, 2)
sentiment
Positive    103059
Neutral      82972
Negative     55114
Name: count, dtype: int64
text         217
sentiment      0
dtype: int64


# Phase 3 and 4 : Data Cleaning and preprocessing

In [12]:
def clean_text(text):
    text=str(text).lower()
    text=re.sub(r"http\S+|www\S+","",text)
    text=re.sub(r"\d+","",text)
    text=text.translate(str.maketrans("","",string.punctuation))
    text=re.sub(r"\s+"," ",text).strip()
    return text

def preprocess_text(text):
    tokens=word_tokenize(text)
    tokens=[t for t in tokens if t not in STOPWORDS]
    tokens=[lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)

def full_pipeline(text):
    return preprocess_text(clean_text(text))

df["processed_text"]=df["text"].apply(full_pipeline)
display(df[["text","processed_text","sentiment"]].head())

,text,processed_text,sentiment
0,lets forget apple pay required brand new iphon...,let forget apple pay required brand new iphone...,Neutral
1,nz retailers don’t even contactless credit car...,nz retailer ’ even contactless credit card mac...,Negative
2,forever acknowledge channel help lessons ideas...,forever acknowledge channel help lesson idea e...,Positive
3,whenever go place doesn’t take apple pay doesn...,whenever go place ’ take apple pay ’ happen of...,Negative
4,apple pay convenient secure easy use used kore...,apple pay convenient secure easy use used kore...,Positive


# Phase 5 : TF-IDF

In [13]:
vectorizer=TfidfVectorizer(max_features=3000)
X=vectorizer.fit_transform(df["processed_text"])
y=df["sentiment"]

print(X.shape)

(241145, 3000)


# Phase 6 : Train Model

In [14]:
X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42,stratify=y
)

model=LogisticRegression(max_iter=1000)
model.fit(X_train,y_train)

print("Training Completed!")

Training Completed!


# Phase 7 : Evaluation

In [15]:
y_pred=model.predict(X_test)

print("Accuracy :",accuracy_score(y_test,y_pred))
print("Precision:",precision_score(y_test,y_pred,average="weighted"))
print("Recall   :",recall_score(y_test,y_pred,average="weighted"))
print("F1 Score :",f1_score(y_test,y_pred,average="weighted"))

print("\nConfusion Matrix")
print(confusion_matrix(y_test,y_pred))

print("\nClassification Report")
print(classification_report(y_test,y_pred))

Accuracy : 0.7715067697858136
Precision: 0.7739610097291583
Recall   : 0.7715067697858136
F1 Score : 0.7697599068659751

Confusion Matrix
[[ 6839  2413  1771]
 [  993 13619  1982]
 [ 1028  2833 16751]]

Classification Report
              precision    recall  f1-score   support

    Negative       0.77      0.62      0.69     11023
     Neutral       0.72      0.82      0.77     16594
    Positive       0.82      0.81      0.81     20612

    accuracy                           0.77     48229
   macro avg       0.77      0.75      0.76     48229
weighted avg       0.77      0.77      0.77     48229



# Phase 8 : Save Model

In [16]:
os.makedirs(BACKEND_DIR,exist_ok=True)

joblib.dump(model,os.path.join(BACKEND_DIR,"model.pkl"))
joblib.dump(vectorizer,os.path.join(BACKEND_DIR,"vectorizer.pkl"))

print("Model Saved Successfully!")
print("backend/model.pkl")
print("backend/vectorizer.pkl")

Model Saved Successfully!
backend/model.pkl
backend/vectorizer.pkl
